# Pipeline

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor

TRAIN_CSV = "../../data/tabular/train/train.csv"
TEST_CSV = "../../data/tabular/test/test.csv"
IMAGE_FOLDER = "../../data/image" 

## Load and preprocess data with ResNet50 to extract features

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# -----------------------------
# Constants
# -----------------------------
TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {
    "Dry_Green_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Clover_g": 0.1,
    "GDM_g": 0.2,
    "Dry_Total_g": 0.5
}

DATASET_DIR = "/kaggle/input/competitions/csiro-biomass"
TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
TEST_CSV = os.path.join(DATASET_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "train")
TEST_IMG_DIR = os.path.join(DATASET_DIR, "test")

# -----------------------------
# Helpers
# -----------------------------
def load_train_data(csv_path):
    df = pd.read_csv(csv_path)
    df["image_id"] = df["sample_id"].str.split("__").str[0]

    df_wide = df.pivot_table(
        index=["image_id", "image_path"],
        columns="target_name",
        values="target"
    ).reset_index()

    y_values = df_wide[TARGETS].values
    return df_wide, y_values

def prepare_submission(test_csv_path, predictions, image_ids):
    df_test = pd.read_csv(test_csv_path)

    pred_dict = {}
    for img_id, pred_vector in zip(image_ids, predictions):
        pred_dict[img_id] = {col: val for col, val in zip(TARGETS, pred_vector)}

    def get_pred(row):
        img_id = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        val = pred_dict.get(img_id, {}).get(target_name, 0.0)
        return max(0.0, float(val))

    df_test["target"] = df_test.apply(get_pred, axis=1)
    return df_test[["sample_id", "target"]]

def weighted_global_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    w = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])

    ybar = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)

def rmse_per_target(y_true, y_pred):
    out = {}
    for i, t in enumerate(TARGETS):
        out[t] = float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
    return out

class ResNetExtractor:
    def __init__(self, device="cpu"):
        self.device = device
        weights = models.ResNet50_Weights.DEFAULT
        resnet = models.resnet50(weights=weights)

        self.model = nn.Sequential(*list(resnet.children())[:-1])
        self.model.to(self.device)
        self.model.eval()

        self.preprocess = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def get_features(self, image_path):
        if not os.path.exists(image_path):
            print(f"Missing file: {image_path}")
            return np.zeros(2048, dtype=np.float32)

        try:
            img = Image.open(image_path).convert("RGB")

            img_left = img.crop((0, 0, 1000, 1000))
            img_right = img.crop((1000, 0, 2000, 1000))

            t_left = self.preprocess(img_left).unsqueeze(0).to(self.device)
            t_right = self.preprocess(img_right).unsqueeze(0).to(self.device)

            with torch.no_grad():
                feat_left = self.model(t_left).squeeze().cpu().numpy()
                feat_right = self.model(t_right).squeeze().cpu().numpy()

            return ((feat_left + feat_right) / 2.0).astype(np.float32)

        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return np.zeros(2048, dtype=np.float32)

# -----------------------------
# Load training data
# -----------------------------
train_df, y_all = load_train_data(TRAIN_CSV)
print("Train images:", len(train_df))
print("Target shape:", y_all.shape)

# -----------------------------
# Extract features for all train images
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
extractor = ResNetExtractor(device=device)

all_train_features = []
all_image_ids = []

for _, row in train_df.iterrows():
    img_path = os.path.join(TRAIN_IMG_DIR, f"{row['image_id']}.jpg")
    feat = extractor.get_features(img_path)
    all_train_features.append(feat)
    all_image_ids.append(row["image_id"])

X_all_raw = np.array(all_train_features, dtype=np.float32)
print("X_all_raw shape:", X_all_raw.shape)

# -----------------------------
# Train / validation split
# -----------------------------
X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
    X_all_raw,
    y_all,
    test_size=0.2,
    random_state=42
)

print("Train split:", X_tr_raw.shape, y_tr.shape)
print("Val split:", X_val_raw.shape, y_val.shape)

# -----------------------------
# PCA fit ONLY on train split
# -----------------------------
pca = PCA(n_components=0.95, random_state=42)
X_tr = pca.fit_transform(X_tr_raw)
X_val = pca.transform(X_val_raw)

print("X_tr PCA shape:", X_tr.shape)
print("X_val PCA shape:", X_val.shape)

# -----------------------------
# Train model
# -----------------------------
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_tr, y_tr)

# -----------------------------
# Validate
# -----------------------------
val_preds = rf.predict(X_val)

print("Weighted global R2:", weighted_global_r2(y_val, val_preds))
print("Overall R2:", r2_score(y_val, val_preds, multioutput="uniform_average"))
print("RMSE per target:")
print(rmse_per_target(y_val, val_preds))

Test it on the test set from kaggle:

In [ ]:
# -----------------------------
# Refit PCA on ALL training data
# -----------------------------
final_pca = PCA(n_components=0.95, random_state=42)
X_all = final_pca.fit_transform(X_all_raw)

print("Final training PCA shape:", X_all.shape)

# -----------------------------
# Refit model on ALL training data
# -----------------------------
final_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
final_rf.fit(X_all, y_all)

# -----------------------------
# Load test data
# -----------------------------
df_test = pd.read_csv(TEST_CSV)
df_test["image_id"] = df_test["sample_id"].str.split("__").str[0]
df_test_unique = df_test.drop_duplicates("image_id").copy()

print("Test images:", len(df_test_unique))

# -----------------------------
# Extract test features
# -----------------------------
test_features = []
test_image_ids = []

for _, row in df_test_unique.iterrows():
    img_path = os.path.join(TEST_IMG_DIR, f"{row['image_id']}.jpg")
    feat = extractor.get_features(img_path)
    test_features.append(feat)
    test_image_ids.append(row["image_id"])

X_test_raw = np.array(test_features, dtype=np.float32)
print("X_test_raw shape:", X_test_raw.shape)

# -----------------------------
# Transform test features with final PCA
# -----------------------------
X_test = final_pca.transform(X_test_raw)

# -----------------------------
# Predict on test
# -----------------------------
test_preds = final_rf.predict(X_test)
print("test_preds shape:", test_preds.shape)

# -----------------------------
# Build submission
# -----------------------------
submission = prepare_submission(TEST_CSV, test_preds, test_image_ids)
print(submission.head())

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved submission to /kaggle/working/submission.csv")